# California County Precipitation Choropleth

This notebook maps county-level January-December 2023 precipitation for California.

In [ ]:
from pathlib import Path
import json
import urllib.request

import pandas as pd
import plotly.express as px

## Load the Precipitation Dataset

In [ ]:
data_candidates = [
    Path("../data/raw/data.csv"),
    Path("data/raw/data.csv"),
    Path("26-the-optimizers-analysis/mini1/data/raw/data.csv"),
]

data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find the precipitation data.csv file")

data_path = data_path.resolve()
mini1_root = data_path.parents[2]

df = pd.read_csv(data_path, comment="#")
df.head()

## Prepare County FIPS Codes

The raw precipitation file uses IDs like `CA-001`. Plotly's county choropleth uses FIPS IDs, so this converts `CA-001` to `06001`.

In [ ]:
map_df = df.copy()

map_df["countyname"] = map_df["Name"].str.replace(" County", "", regex=False).str.strip()
map_df["fips"] = "06" + map_df["ID"].str.extract(r"CA-(\d{3})", expand=False)

numeric_columns = [
    "Value",
    "Anomaly (1901-2000 base period)",
    "Rank",
    "1901-2000 Mean",
    "Pctavg",
    "Percent of Average",
]

for column in numeric_columns:
    map_df[column] = pd.to_numeric(map_df[column], errors="coerce")

missing_fips = sorted(map_df.loc[map_df["fips"].isna(), "Name"].dropna().unique())
if missing_fips:
    raise ValueError(f"Could not create FIPS codes for: {missing_fips}")

map_df.head()

## Build the Choropleth Map

In [ ]:
ca_geojson_path = mini1_root / "data/interim/ca_counties.geojson"

if ca_geojson_path.exists():
    with ca_geojson_path.open(encoding="utf-8") as file:
        ca_counties_geojson = json.load(file)
else:
    counties_geojson_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

    try:
        with urllib.request.urlopen(counties_geojson_url) as response:
            counties_geojson = json.load(response)
    except Exception as exc:
        raise RuntimeError(
            "Could not download county GeoJSON. Connect to the internet and rerun this cell."
        ) from exc

    ca_counties_geojson = {
        "type": "FeatureCollection",
        "features": [
            feature
            for feature in counties_geojson["features"]
            if feature["id"].startswith("06")
        ],
    }
    ca_geojson_path.write_text(json.dumps(ca_counties_geojson), encoding="utf-8")

fig = px.choropleth(
    map_df,
    geojson=ca_counties_geojson,
    locations="fips",
    color="Value",
    color_continuous_scale="Blues",
    hover_name="countyname",
    hover_data={
        "fips": False,
        "ID": False,
        "State": False,
        "Value": ":.2f",
        "Anomaly (1901-2000 base period)": ":.2f",
        "Rank": ":.0f",
        "1901-2000 Mean": ":.2f",
        "Percent of Average": ":.1f",
    },
    labels={
        "Value": "Precipitation (inches)",
        "Anomaly (1901-2000 base period)": "Anomaly (inches)",
        "1901-2000 Mean": "1901-2000 mean (inches)",
        "Percent of Average": "Percent of average",
    },
    title="2023 Precipitation by California County",
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 50},
    coloraxis_colorbar_title="Precipitation<br>(inches)",
)

fig.show()

## Save an HTML Copy

In [ ]:
output_path = mini1_root / "results/figures/precipitation_choropleth.html"
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(output_path)
output_path